# EDL Full Experiment (parallel)

10-fold cross-validation comparison of 8 LDL models across 3 datasets
(`SJAFFE`, `SBU_3DFE`, `Human_Gene`), parallelised across CPU workers or
GPUs via `loky`.

The actual training loop lives in `edl_workers.py` next to this notebook
so loky subprocesses can `import edl_workers` cleanly. TensorFlow is
imported lazily inside each worker after `CUDA_VISIBLE_DEVICES` is pinned.

For each (model, dataset) pair we record six distributional metrics
(`chebyshev`, `clark`, `canberra`, `kl_divergence`, `cosine`, `intersection`)
across 10 folds with a 10% test split per fold, then summarise as
mean ± std.

For the evidential models (`EDL_LDL`, `BEDL_LDL`) and `SNEFY_LDL` we also
report:

- **Mean uncertainty** — average per-sample uncertainty on the test set
  (lower = the model is more confident).
- **Uncertainty calibration (Spearman ρ)** — rank correlation between
  per-sample uncertainty and per-sample KL divergence error.
  Higher = uncertainty tracks error better.


In [12]:
!pip install tensorflow-probability

Defaulting to user installation because normal site-packages is not writeable
Looking in links: /cvmfs/soft.computecanada.ca/custom/python/wheelhouse/gentoo2023/x86-64-v4, /cvmfs/soft.computecanada.ca/custom/python/wheelhouse/gentoo2023/x86-64-v3, /cvmfs/soft.computecanada.ca/custom/python/wheelhouse/gentoo2023/generic, /cvmfs/soft.computecanada.ca/custom/python/wheelhouse/generic


In [13]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import multiprocessing as mp
from collections import defaultdict
from concurrent.futures import as_completed

import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from loky import get_reusable_executor

from pyldl.utils import load_dataset

import demo.edl_workers
from demo.edl_workers import init_worker, run_one_fold, MODEL_NAMES, METRICS


## Configuration

`GPU_IDS` — list of CUDA device ids to use, one worker per id.
Set to `[]` for CPU-only; in that case `N_WORKERS` controls the pool size.

`N_EPOCHS` is passed to every model whose `fit()` accepts it (everything
except `SA_BFGS`).


In [3]:
DATASETS = ['SJAFFE'] #, 'SBU_3DFE', 'Human_Gene']
N_SPLITS = 2
N_EPOCHS = 100
RANDOM_STATE = 0

# --- Parallel config -----------------------------------------------------
GPU_IDS = []                                 # e.g. [0, 1, 2, 3] for 4 GPUs
N_WORKERS = len(GPU_IDS) if GPU_IDS else max(1, (os.cpu_count() or 2) // 2)
# -------------------------------------------------------------------------


## Build the worker pool

Each worker pulls one entry off `gpu_queue` exactly once at startup
(loky's `initializer`) and pins `CUDA_VISIBLE_DEVICES` before TF sees a
GPU. `reuse=False` forces a fresh pool if you re-run this cell after
changing the config.


In [4]:
mgr = mp.Manager()
gpu_queue = mgr.Queue()

slots = list(GPU_IDS) if GPU_IDS else [None] * N_WORKERS
assert len(slots) == N_WORKERS, 'one queue slot per worker'
for g in slots:
    gpu_queue.put(g)

executor = get_reusable_executor(
    max_workers=N_WORKERS,
    initializer=init_worker,
    initargs=(gpu_queue,),
    reuse=False,
)
print(f'pool ready: {N_WORKERS} workers, gpu_ids={slots}')


pool ready: 24 workers, gpu_ids=[None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None]


## Build the job list

`KFold` splits are generated in the parent (deterministic, cheap) and the
fold slices are passed to workers as numpy arrays. Loky memmaps large
numpy arrays automatically, so this is fast even for the bigger datasets.


In [6]:
jobs = []
for dataset_name in DATASETS:
    X, D = load_dataset(dataset_name, dir='dataset')
    kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    for fold_idx, (train_idx, test_idx) in enumerate(kf.split(X), start=1):
        Xtr, Xte = X[train_idx], X[test_idx]
        Dtr, Dte = D[train_idx], D[test_idx]
        for model_name in MODEL_NAMES:
            jobs.append((dataset_name, model_name, fold_idx, Xtr, Dtr, Xte, Dte))

total = len(jobs)
print(f'queued {total} jobs ({len(DATASETS)} datasets × {N_SPLITS} folds × {len(MODEL_NAMES)} models)')


queued 16 jobs (1 datasets × 2 folds × 8 models)


In [14]:
import time

# --- Sanity-check config -------------------------------------------------
PROBE_N    = N_WORKERS * 2   # number of jobs to time on each side
PROBE_EPOCHS = 20            # smaller than N_EPOCHS so the test finishes quickly
# -------------------------------------------------------------------------

probe_jobs = jobs[:PROBE_N]
print(f'timing {PROBE_N} jobs ({PROBE_EPOCHS} epochs each) on {N_WORKERS} workers...\n')

# --- Serial: run in the parent process -----------------------------------
t0 = time.time()
serial_results = [
    run_one_fold(ds, m, fi, Xtr, Dtr, Xte, Dte, PROBE_EPOCHS)
    for (ds, m, fi, Xtr, Dtr, Xte, Dte) in probe_jobs
]
serial_time = time.time() - t0
print(f'serial:   {serial_time:6.1f}s   ({serial_time / PROBE_N:.1f}s per job)')

# --- Parallel: dispatch to the loky pool ---------------------------------
t0 = time.time()
futs = [
    executor.submit(run_one_fold, ds, m, fi, Xtr, Dtr, Xte, Dte, PROBE_EPOCHS)
    for (ds, m, fi, Xtr, Dtr, Xte, Dte) in probe_jobs
]
parallel_results = [f.result() for f in futs]
parallel_time = time.time() - t0
print(f'parallel: {parallel_time:6.1f}s   ({parallel_time / PROBE_N:.1f}s per job)')

# --- Verdict --------------------------------------------------------------
speedup = serial_time / parallel_time
ideal   = min(N_WORKERS, PROBE_N)
efficiency = speedup / ideal
print(f'\nspeedup:    {speedup:.2f}x   (ideal: {ideal:.0f}x)')
print(f'efficiency: {efficiency * 100:.0f}%')

if efficiency < 0.4:
    print('\n  speedup is low — likely causes:')
    print('   - all workers landed on the same GPU (check nvidia-smi)')
    print('   - workers are CPU-bound and oversubscribed (lower N_WORKERS or '
          'inter/intra_op_threads in init_worker)')
    print('   - cold-start dominates: PROBE_N is too small relative to N_WORKERS')
elif efficiency < 0.7:
    print('\n  decent speedup; some overhead from worker cold-start or IPC')
else:
    print('\n  good parallel scaling')


timing 48 jobs (20 epochs each) on 24 workers...



I0000 00:00:1777054137.838429 1084351 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 8074 MB memory:  -> device: 0, name: NVIDIA H100 80GB HBM3 MIG 1g.10gb, pci bus id: 0000:0c:00.0, compute capability: 9.0


serial:     51.7s   (1.1s per job)


E0000 00:00:1777054188.790536 1087519 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777054188.798506 1087519 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
E0000 00:00:1777054188.842518 1087512 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
W0000 00:00:1777054188.851649 1087519 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777054188.851674 1087519 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777054188.851677 1087519 computation_placer.cc:177] computation placer already registered. Please check linka

parallel:   54.7s   (1.1s per job)

speedup:    0.94x   (ideal: 24x)
efficiency: 4%

  speedup is low — likely causes:
   - all workers landed on the same GPU (check nvidia-smi)
   - workers are CPU-bound and oversubscribed (lower N_WORKERS or inter/intra_op_threads in init_worker)
   - cold-start dominates: PROBE_N is too small relative to N_WORKERS


## Submit + collect

Submission is non-blocking; results stream back via `as_completed` so the
log shows progress as folds finish. Per-fold failures are caught and
printed but don't stop the run.


In [29]:
futures = {
    executor.submit(run_one_fold, ds, m, fi, Xtr, Dtr, Xte, Dte, N_EPOCHS): (ds, m, fi)
    for (ds, m, fi, Xtr, Dtr, Xte, Dte) in jobs
}

raw_results = []
for i, fut in enumerate(as_completed(futures), start=1):
    ds, m, fi = futures[fut]
    try:
        raw_results.append(fut.result())
        status = 'ok'
    except Exception as e:
        status = f'FAILED ({type(e).__name__}: {e})'
    print(f'[{i:4d}/{total}] {ds:12s} | fold {fi:2d} | {m:30s} {status}')


Exception in initializer:
Traceback (most recent call last):
  File "/home/dcs01/.local/lib/python3.12/site-packages/loky/process_executor.py", line 434, in _process_worker
    initializer(*initargs)
  File "/project/6004619/dcs01/PyLDL/demo/edl_workers.py", line 26, in init_worker
    gpu_id = gpu_queue.get()
             ^^^^^^^^^^^^^^^
  File "<string>", line 2, in get
  File "/cvmfs/soft.computecanada.ca/easybuild/software/2023/x86-64-v4/Compiler/gcccore/python/3.12.4/lib/python3.12/multiprocessing/managers.py", line 821, in _callmethod
    kind, result = conn.recv()
                   ^^^^^^^^^^^
  File "/cvmfs/soft.computecanada.ca/easybuild/software/2023/x86-64-v4/Compiler/gcccore/python/3.12.4/lib/python3.12/multiprocessing/connection.py", line 250, in recv
    buf = self._recv_bytes()
          ^^^^^^^^^^^^^^^^^^
  File "/cvmfs/soft.computecanada.ca/easybuild/software/2023/x86-64-v4/Compiler/gcccore/python/3.12.4/lib/python3.12/multiprocessing/connection.py", line 430, in _rec

KeyboardInterrupt: 

## Bucket results into per-model DataFrames

`per_model_results[(dataset, model_name)]` is a DataFrame with one row per
fold; columns are the recorded metrics.


In [22]:
buckets = defaultdict(list)
for r in raw_results:
    buckets[(r['dataset'], r['model'])].append(r['scores'])

per_model_results = {key: pd.DataFrame(rows) for key, rows in buckets.items()}


## Per-model fold tables

Inspect any single (dataset, model) DataFrame:


In [24]:
per_model_results

{}

In [23]:
per_model_results[('SJAFFE', 'EDL_LDL (loglikelihood)')]


KeyError: ('SJAFFE', 'EDL_LDL (loglikelihood)')

## Combined summary — mean ± std across folds

One row per (dataset, model); columns are `metric_mean` / `metric_std`.


In [ ]:
def summarize(df):
    out = {}
    for col in df.columns:
        out[f'{col}_mean'] = df[col].mean()
        out[f'{col}_std']  = df[col].std()
    return out


summary_rows = []
for (dataset_name, model_name), df in per_model_results.items():
    if df.empty:
        continue
    row = {'dataset': dataset_name, 'model': model_name, **summarize(df)}
    summary_rows.append(row)

summary = pd.DataFrame(summary_rows).set_index(['dataset', 'model'])
summary


### Compact view: `mean ± std` per metric


In [ ]:
def fmt(mean, std):
    if pd.isna(mean):
        return ''
    return f'{mean:.4f} ± {std:.4f}'


compact_rows = []
for (dataset_name, model_name), df in per_model_results.items():
    if df.empty:
        continue
    row = {'dataset': dataset_name, 'model': model_name}
    for col in df.columns:
        row[col] = fmt(df[col].mean(), df[col].std())
    compact_rows.append(row)

compact = pd.DataFrame(compact_rows).set_index(['dataset', 'model'])
compact


## Uncertainty results (EDL_LDL, BEDL_LDL, SNEFY_LDL only)

- `mean_uncertainty` — average per-sample uncertainty on test (model-specific
  scale; lower = more confident).
- `uncertainty_calibration` — Spearman ρ between per-sample uncertainty and
  per-sample KL divergence error. Higher = uncertainty better predicts error.


In [ ]:
uncertainty_models = {
    'EDL_LDL (loglikelihood)', 'EDL_LDL (bayes_mse)',
    'BEDL_LDL (loglikelihood)', 'BEDL_LDL (bayes_mse)',
    'SNEFY_LDL',
}

uncertainty_rows = []
for (dataset_name, model_name), df in per_model_results.items():
    if model_name not in uncertainty_models or df.empty:
        continue
    if 'mean_uncertainty' not in df.columns:
        continue
    uncertainty_rows.append({
        'dataset': dataset_name,
        'model': model_name,
        'mean_uncertainty':        fmt(df['mean_uncertainty'].mean(),        df['mean_uncertainty'].std()),
        'uncertainty_calibration': fmt(df['uncertainty_calibration'].mean(), df['uncertainty_calibration'].std()),
    })

uncertainty_summary = pd.DataFrame(uncertainty_rows).set_index(['dataset', 'model'])
uncertainty_summary


## Shut down the pool

Loky reuses pools by default; close it explicitly when you're done so the
worker processes (and any GPU memory they hold) are released.


In [ ]:
executor.shutdown(wait=True, kill_workers=True)
